# Learning curves and grouped statistical evaluation

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load development cohort

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Train on increasing fractions of unique patients in each fold.

In [ ]:
from src.advanced_experiment import pipeline
from sklearn.metrics import average_precision_score
rows=[]
for i,(t,v) in enumerate(folds(X,Y,G)):
    ids=G.iloc[t].unique().copy(); np.random.default_rng(42+i).shuffle(ids)
    for frac in [.1,.25,.5,.75,1.]:
        tt=t[G.iloc[t].isin(ids[:max(1,int(len(ids)*frac))]).to_numpy()]; m=pipeline(X,QUICK).fit(X.iloc[tt],Y.iloc[tt]); rows.append({'fold':i,'fraction':frac,'train_ap':average_precision_score(Y.iloc[tt],m.predict_proba(X.iloc[tt])[:,1]),'validation_ap':average_precision_score(Y.iloc[v],m.predict_proba(X.iloc[v])[:,1])})
d=pd.DataFrame(rows); display(d); d.groupby('fraction')[['train_ap','validation_ap']].mean().plot(marker='o'); plt.ylabel('Average precision'); plt.show(); d.to_csv(OUT/'learning_curves.csv',index=False)

Patient-level bootstrap comparison on validation data

In [ ]:
from src.evaluation.compare import compare
from src.models.boosting import models
preds={}
for k,est in models().items():
    m=make_model(X,est,quick=QUICK).fit(X,Y); preds[k]=m.predict_proba(x.iloc[tune])[:,1]
winner=max(preds,key=lambda k:average_precision_score(y.iloc[tune],preds[k])); compare(y.iloc[tune].to_numpy(),g.iloc[tune].to_numpy(),preds,winner,OUT,30 if QUICK else 300); display(pd.read_csv(OUT/'patient_bootstrap.csv'))